# Notebook 03 — Churn Prediction Model

Logistic Regression and Decision Tree trained on customer features. Includes coefficients, tree visualization, confusion matrix, and ROC curves.

In [1]:
import sys
sys.path.insert(0, '..')
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              confusion_matrix, roc_auc_score, roc_curve,
                              ConfusionMatrixDisplay)
from skills.utils.plotting import save_fig
from skills.utils.churn_scoring import score_customers, top_churn_features

sns.set_theme(style='whitegrid')


## Load & Prepare Features

In [2]:
df = pd.read_csv('../Customer-Churn-Records.csv')
df_model = df.drop(columns=['RowNumber', 'CustomerId', 'Surname'])
df_encoded = pd.get_dummies(df_model, columns=['Geography', 'Gender', 'Card Type'], drop_first=False)

X = df_encoded.drop(columns=['Exited'])
y = df_encoded['Exited']
feature_names = X.columns.tolist()

print(f"Features ({len(feature_names)}): {feature_names}")
print(f"Class balance: {y.value_counts().to_dict()}")


Features (20): ['CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary', 'Complain', 'Satisfaction Score', 'Point Earned', 'Geography_France', 'Geography_Germany', 'Geography_Spain', 'Gender_Female', 'Gender_Male', 'Card Type_DIAMOND', 'Card Type_GOLD', 'Card Type_PLATINUM', 'Card Type_SILVER']
Class balance: {0: 7962, 1: 2038}


## Train / Test Split

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {len(X_train)} rows | Test: {len(X_test)} rows")
print(f"Train churn rate: {y_train.mean():.1%} | Test churn rate: {y_test.mean():.1%}")


Train: 8000 rows | Test: 2000 rows
Train churn rate: 20.4% | Test churn rate: 20.4%


## Logistic Regression

In [4]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_scaled, y_train)

y_pred_lr = lr.predict(X_test_scaled)
y_prob_lr = lr.predict_proba(X_test_scaled)[:, 1]

print("=== Logistic Regression Performance ===")
print(f"Accuracy:  {accuracy_score(y_test, y_pred_lr):.3f}")
print(f"Precision: {precision_score(y_test, y_pred_lr):.3f}")
print(f"Recall:    {recall_score(y_test, y_pred_lr):.3f}")
print(f"ROC-AUC:   {roc_auc_score(y_test, y_prob_lr):.3f}")


=== Logistic Regression Performance ===
Accuracy:  0.999
Precision: 0.998
Recall:    0.995
ROC-AUC:   0.999


### Logistic Regression Coefficients

Positive coefficients increase churn risk; negative coefficients decrease it.

In [5]:
coef_df = pd.DataFrame({
    'feature': feature_names,
    'coefficient': lr.coef_[0]
}).sort_values('coefficient', ascending=False)

fig, ax = plt.subplots(figsize=(10, 8))
colors = ['#E53935' if c > 0 else '#1E88E5' for c in coef_df['coefficient']]
ax.barh(coef_df['feature'], coef_df['coefficient'], color=colors, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Logistic Regression Coefficients\n(red = increases churn risk, blue = decreases)', fontsize=12)
ax.set_xlabel('Coefficient Value')
plt.tight_layout()
save_fig(fig, '03_lr_coefficients.png', output_dir='../outputs/figures')
plt.close(fig)

coef_df.to_csv('../outputs/tables/lr_coefficients.csv', index=False)
print(coef_df.to_string(index=False))


           feature  coefficient
          Complain     5.183796
               Age     0.854531
  Card Type_SILVER     0.109805
  Geography_France     0.072552
       CreditScore     0.065510
           Balance     0.038919
   Geography_Spain     0.035494
   EstimatedSalary     0.033255
       Gender_Male     0.031372
 Card Type_DIAMOND    -0.000972
     Gender_Female    -0.031372
         HasCrCard    -0.041883
Card Type_PLATINUM    -0.053670
    Card Type_GOLD    -0.054595
            Tenure    -0.065481
     NumOfProducts    -0.085284
 Geography_Germany    -0.118994
Satisfaction Score    -0.208162
      Point Earned    -0.395773
    IsActiveMember    -0.621829


## Decision Tree

In [6]:
dt = DecisionTreeClassifier(max_depth=4, random_state=42, class_weight='balanced')
dt.fit(X_train, y_train)

y_pred_dt = dt.predict(X_test)
y_prob_dt = dt.predict_proba(X_test)[:, 1]

print("=== Decision Tree Performance ===")
print(f"Accuracy:  {accuracy_score(y_test, y_pred_dt):.3f}")
print(f"Precision: {precision_score(y_test, y_pred_dt):.3f}")
print(f"Recall:    {recall_score(y_test, y_pred_dt):.3f}")
print(f"ROC-AUC:   {roc_auc_score(y_test, y_prob_dt):.3f}")


=== Decision Tree Performance ===
Accuracy:  0.998
Precision: 0.995
Recall:    0.995
ROC-AUC:   0.997


### Decision Tree Visualization

In [7]:
fig, ax = plt.subplots(figsize=(24, 10))
plot_tree(dt, feature_names=feature_names, class_names=['Stay', 'Churn'],
          filled=True, rounded=True, ax=ax, fontsize=8, impurity=False)
ax.set_title('Decision Tree (max_depth=4)', fontsize=13)
save_fig(fig, '03_decision_tree.png', output_dir='../outputs/figures')
plt.close(fig)


### Feature Importances (Decision Tree)

In [8]:
fi_df = pd.DataFrame({
    'feature': feature_names,
    'importance': dt.feature_importances_
}).sort_values('importance', ascending=False).head(10)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(fi_df['feature'], fi_df['importance'], color='steelblue', edgecolor='white')
ax.set_title('Top 10 Feature Importances (Decision Tree)', fontsize=12)
ax.set_xlabel('Importance')
plt.tight_layout()
save_fig(fig, '03_feature_importance.png', output_dir='../outputs/figures')
plt.close(fig)

fi_df.to_csv('../outputs/tables/feature_importance.csv', index=False)
print(fi_df.to_string(index=False))


           feature  importance
          Complain    0.996899
      Point Earned    0.001294
   EstimatedSalary    0.001233
            Tenure    0.000311
 Geography_Germany    0.000251
               Age    0.000008
           Balance    0.000004
   Geography_Spain    0.000000
Card Type_PLATINUM    0.000000
    Card Type_GOLD    0.000000


## Confusion Matrices

In [9]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, name, y_pred in zip(axes,
                             ['Logistic Regression', 'Decision Tree'],
                             [y_pred_lr, y_pred_dt]):
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=['Stay', 'Churn'])
    disp.plot(ax=ax, colorbar=False)
    ax.set_title(name, fontsize=12)
plt.suptitle('Confusion Matrices', fontsize=14)
plt.tight_layout()
save_fig(fig, '03_confusion_matrices.png', output_dir='../outputs/figures')
plt.close(fig)


## ROC Curves

In [10]:
fpr_lr, tpr_lr, _ = roc_curve(y_test, y_prob_lr)
fpr_dt, tpr_dt, _ = roc_curve(y_test, y_prob_dt)
auc_lr = roc_auc_score(y_test, y_prob_lr)
auc_dt = roc_auc_score(y_test, y_prob_dt)

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(fpr_lr, tpr_lr, label=f'Logistic Regression (AUC={auc_lr:.3f})', linewidth=2)
ax.plot(fpr_dt, tpr_dt, label=f'Decision Tree (AUC={auc_dt:.3f})', linewidth=2, linestyle='--')
ax.plot([0, 1], [0, 1], 'k--', alpha=0.4, label='Random')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — Churn Prediction Models', fontsize=12)
ax.legend()
plt.tight_layout()
save_fig(fig, '03_roc_curves.png', output_dir='../outputs/figures')
plt.close(fig)
